# Claude Code provider: the harness parts, one at a time

`ClaudeCodeProvider` (`agent_base/providers/claude_code/`) sends a session's model calls through **Claude Code**, the Claude Agent SDK's CLI, instead of straight to the Anthropic API. Only the model call moves: the turn loop still runs every tool, pauses for frontend tools, checkpoints and settles, as on the direct client (`AnthropicProvider`), which the provider wraps and falls back to.

A host plugs it in with one argument:

```python
AnthropicAgent(..., provider_value=ClaudeCodeProvider(config=ClaudeCodeConfig(workdir="/var/claude_code")))
```

**Offline first.** Sections 1 to 13 drive the provider against a scripted stand-in for Claude Code: no CLI process, no database, no model call. The stand-in replays raw Anthropic stream events and calls tools through the SDK's real in-process MCP server, as the CLI does. Section 14 is an opt-in live run on your own Claude Code sign-in. Needs the `claude-code` extra (`uv sync --extra claude-code`).

| Piece | What it does | File |
|---|---|---|
| `ClaudeCodeProvider` | The `Provider` protocol over one run's Claude Code process, plus the optional lifecycle | `provider.py` |
| `ClaudeCodeConfig` | Working folder, MCP server name, fallback mode, web tools, URL guard, process cap | `config.py` |
| Fingerprint | A hash of the history up to a point, stable across a save and a cold load | `fingerprint.py` |
| Transcript store, ledger | Claude Code's transcripts, and the map from points in the library's history to them | `transcripts.py`; Postgres in `agent_base/storage/pg/transcripts.py` |
| Process slots, the plan's meter | How many CLI processes run at once; the plan's last rate-limit state | `slots.py`, `meter.py` |
| URL guard | Which URLs Claude Code's `WebFetch` may open from the host | `url_guard.py` |

```
 turn loop ──generate_stream(full history)──▶ ClaudeCodeProvider ──query / receive_messages──▶ Claude Code CLI ──▶ model
     ▲                                            │      ▲                                           │
     │◀── ProviderTurn, frames on the sink ───────┘      └──── MCP handler waits for its result ◀────┘ tool call
     └── tool round runs the tools, appends results ──▶ next generate_stream hands each waiting handler its result
```

In [1]:
from dotenv import load_dotenv
import asyncio
import importlib.metadata
import json
import os
import tempfile
import uuid
from pathlib import Path

load_dotenv()

import claude_agent_sdk
from agent_base.providers.claude_code import (ClaudeCodeConfig, ClaudeCodeProvider, MemoryTranscriptLedger,
                                              MemoryTranscriptStore, PlanMeter, ProcessSlots, cli_env,
                                              prefix_fingerprints)
from agent_base.providers.claude_code.provider import _plain_history

print("claude-agent-sdk :", claude_agent_sdk.__version__)
print("mcp              :", importlib.metadata.version("mcp"), "(below 2: the provider reads each tool call's id from mcp 1.x)")

claude-agent-sdk : 0.2.180
mcp              : 1.28.1 (below 2: the provider reads each tool call's id from mcp 1.x)


In [2]:
# ── Test Utils ─────────────────────────────────────────────────────
#
# FakeClaudeCode stands in for claude_agent_sdk.ClaudeSDKClient (connect / query / receive_messages / disconnect).
# Each script entry is one model call: a list of blocks, or a failure(...) the CLI reports. A model call with tool calls makes the fake call each
# tool through the provider's real in-process MCP server, with the model's tool-call id in `_meta` as the CLI
# sends it, and stream the next model call once every result is back. Claude Code's own WebSearch / WebFetch the
# fake "runs" itself. It mirrors each transcript entry into the session store, as the CLI does.

from types import SimpleNamespace

import anthropic
import mcp.types as mcp_types
from claude_agent_sdk import AssistantMessage, ResultMessage, StreamEvent, TextBlock, ToolResultBlock, UserMessage
from claude_agent_sdk._internal.session_store import project_key_for_directory
from mcp.server.lowlevel.server import request_ctx

from agent_base.core.messages import Message
from agent_base.core.provider import RetryPolicy
from agent_base.core.types import (Role, ServerToolResultContent, ServerToolUseContent, TextContent,
                                   ToolResultContent, ToolUseContent)
from agent_base.providers.anthropic import AnthropicAgent, AnthropicLLMConfig
from agent_base.providers.anthropic.provider import AnthropicProvider
from agent_base.tools import ToolRegistry, tool

MODEL = "claude-opus-5-5"
WORKDIR = Path(tempfile.gettempdir()) / "claude_code_notebook"
CC_WEB = {"WebSearch", "WebFetch"}
_n = iter(range(1, 100_000))


def text(t):
    return {"type": "text", "text": t}


def call(name, **args):
    """One of the library's tools: Claude Code names it mcp__agent_base__<tool>."""
    return {"type": "tool_use", "id": f"toolu_{next(_n)}", "name": f"mcp__agent_base__{name}", "input": args}


def web(name, result, **args):
    """Claude Code's own WebSearch / WebFetch, with the result the CLI gets back."""
    return {"type": "tool_use", "id": f"srvtoolu_{next(_n)}", "name": name, "input": args, "result": result}


def failure(said, *, error="invalid_request", reason=None, status=None, subtype="success"):
    """A model call the CLI reports as failed, as CLI 2.1.321 does: its own assistant message carrying `error`, then
    the query's result with is_error. `reason` is the result's terminal_reason (prompt_too_long, blocking_limit, ...);
    a subtype other than "success" is the CLI ending the query itself (error_max_turns)."""
    return {"fail": dict(said=said, error=error, reason=reason, status=status, subtype=subtype)}


def cut_reply(blocks, partial="", *, resumed_ok=False):
    """A reply cut off mid-stream, as CLI 2.1.321 handles one: `blocks` finish, `partial` starts a text block that
    never does, and no message_stop. The CLI keeps the finished blocks and sends its own error message; then it runs
    their tool calls, or asks the model to resume (the script's next entry)."""
    return {"cut": dict(blocks=blocks, partial=partial, resumed_ok=resumed_ok)}


def model_call(msg_id, blocks, stop, *, session_id, out_tokens=10, cache_1h=100):
    usage = {"input_tokens": 5, "output_tokens": 1, "cache_creation_input_tokens": cache_1h,
             "cache_read_input_tokens": 50,
             "cache_creation": {"ephemeral_5m_input_tokens": 0, "ephemeral_1h_input_tokens": cache_1h}}
    ev = [{"type": "message_start", "message": {"id": msg_id, "type": "message", "role": "assistant", "content": [],
                                                "model": MODEL, "stop_reason": None, "stop_sequence": None,
                                                "usage": usage}}]
    for i, b in enumerate(blocks):
        if b["type"] == "text":
            ev += [{"type": "content_block_start", "index": i, "content_block": {"type": "text", "text": ""}},
                   {"type": "content_block_delta", "index": i, "delta": {"type": "text_delta", "text": b["text"]}}]
        else:
            ev += [{"type": "content_block_start", "index": i,
                    "content_block": {"type": "tool_use", "id": b["id"], "name": b["name"], "input": {}}},
                   {"type": "content_block_delta", "index": i,
                    "delta": {"type": "input_json_delta", "partial_json": json.dumps(b["input"])}}]
        ev.append({"type": "content_block_stop", "index": i})
    ev += [{"type": "message_delta", "delta": {"stop_reason": stop, "stop_sequence": None},
            "usage": {"output_tokens": out_tokens}}, {"type": "message_stop"}]
    out = [StreamEvent(uuid=f"ev-{msg_id}-{i}", session_id=session_id, event=e) for i, e in enumerate(ev)]
    out.append(AssistantMessage(content=[], model=MODEL, message_id=msg_id, uuid=f"entry-{msg_id}",
                                session_id=session_id))
    return out


class FakeClaudeCode:
    made: list = []

    def __init__(self, opts, script=()):
        self.opts, self.script = opts, list(script)
        self.sent, self.tool_calls, self.closed = [], [], False
        self.q = asyncio.Queue()
        self.session_id = opts.resume if (opts.resume and not opts.fork_session) else f"cc-{uuid.uuid4().hex[:8]}"
        self.project_key = project_key_for_directory(str(opts.cwd))
        FakeClaudeCode.made.append(self)
        self.open_at_start = sum(not f.closed for f in FakeClaudeCode.made)  # processes open, this one included

    async def connect(self):
        return None

    async def query(self, prompt):
        async for m in prompt:
            self.sent.append(m["message"]["content"])
        await self.next_model_call()

    async def receive_messages(self):
        while True:
            m = await self.q.get()
            if m is None:
                return
            if isinstance(m, BaseException):
                raise m
            yield m

    async def disconnect(self):
        self.closed = True
        self.q.put_nowait(None)

    def push(self, *items):
        for m in items:
            self.q.put_nowait(m)

    async def next_model_call(self):
        if not self.script:
            return
        blocks = self.script.pop(0)
        if isinstance(blocks, dict) and "cut" in blocks:
            await self._cut(**blocks["cut"])
            return
        if isinstance(blocks, dict):
            f = blocks["fail"]
            if f["error"]:
                self.push(AssistantMessage(content=[TextBlock(text=f["said"])], model="<synthetic>", error=f["error"],
                                           message_id=f"msg_{next(_n)}", uuid=f"entry-err-{next(_n)}",
                                           session_id=self.session_id))
            ok = f["subtype"] == "success"
            self.push(ResultMessage(subtype=f["subtype"], duration_ms=1, duration_api_ms=1, is_error=True, num_turns=1,
                                    session_id=self.session_id, result=f["said"] if ok else None,
                                    errors=None if ok else [f["said"]], api_error_status=f["status"],
                                    terminal_reason=f["reason"]))
            return
        msg_id = f"msg_{next(_n)}"
        uses = [b for b in blocks if b["type"] == "tool_use"]
        wire = [{k: v for k, v in b.items() if k != "result"} for b in blocks]
        if self.opts.session_store is not None:
            await self.opts.session_store.append({"project_key": self.project_key, "session_id": self.session_id},
                                                 [{"type": "assistant", "uuid": f"entry-{msg_id}"}])
        self.push(*model_call(msg_id, wire, "tool_use" if uses else "end_turn", session_id=self.session_id))
        if not uses:
            self.push(ResultMessage(subtype="success", duration_ms=1, duration_api_ms=1, is_error=False,
                                    num_turns=1, session_id=self.session_id))
            return
        mine = [b for b in uses if b["name"] in CC_WEB]
        if mine:
            self.push(UserMessage(content=[ToolResultBlock(tool_use_id=b["id"], content=b["result"]) for b in mine]))
        theirs = [b for b in uses if b["name"] not in CC_WEB]
        if theirs:
            self.tool_calls.append(asyncio.create_task(self._run_tools(theirs)))
        else:
            await self.next_model_call()

    async def _cut(self, blocks, partial, resumed_ok):
        msg_id = f"msg_{next(_n)}"
        events = model_call(msg_id, blocks + ([text(partial)] if partial else []), "end_turn",
                            session_id=self.session_id)[:-1]
        events = [e for e in events if e.event["type"] not in ("message_delta", "message_stop")]
        if partial:  # the last block never finishes
            del events[max(i for i, e in enumerate(events) if e.event["type"] == "content_block_stop")]
        self.push(*events)
        if self.opts.session_store is not None:
            await self.opts.session_store.append({"project_key": self.project_key, "session_id": self.session_id},
                                                 [{"type": "assistant", "uuid": f"entry-{msg_id}"}])
        self.push(AssistantMessage(content=[], model=MODEL, message_id=msg_id, uuid=f"entry-{msg_id}",
                                   session_id=self.session_id),
                  AssistantMessage(content=[TextBlock(text="API Error: Server error mid-response.")],
                                   model="<synthetic>", error="server_error", message_id=f"msg_{next(_n)}",
                                   uuid=f"entry-err-{next(_n)}", session_id=self.session_id))
        uses = [b for b in blocks if b["type"] == "tool_use"]
        if uses:
            self.tool_calls.append(asyncio.create_task(self._run_tools(uses)))
        elif self.script:
            await self.next_model_call()  # the model resumes
        else:
            self.push(ResultMessage(subtype="success", duration_ms=1, duration_api_ms=1, is_error=not resumed_ok,
                                    num_turns=1, session_id=self.session_id,
                                    result=None if resumed_ok else "API Error: Server error mid-response."))

    async def _run_tools(self, uses):
        server = next(iter(self.opts.mcp_servers.values()))["instance"]
        results = await asyncio.gather(*(self.call_tool(server, u) for u in uses))
        await self.next_model_call()
        return results

    async def call_tool(self, server, use):
        params = mcp_types.CallToolRequestParams(name=use["name"].split("__", 2)[-1], arguments=use["input"],
                                                 _meta={"claudecode/toolUseId": use["id"]})
        req = mcp_types.CallToolRequest(method="tools/call", params=params)
        token = request_ctx.set(SimpleNamespace(meta=params.meta, request_id=use["id"], session=None,
                                                lifespan_context=None, request=None))
        try:
            return await server.request_handlers[mcp_types.CallToolRequest](req)
        finally:
            request_ctx.reset(token)


class Sink:
    def __init__(self):
        self.deltas = []

    def emit(self, d):
        self.deltas.append(d)

    def emit_meta(self, body):
        pass


def render(d):
    t = getattr(d, "type", "")
    if t == "text":
        return f"text {d.text!r}" if d.text else "text (block closed)"
    if t in ("tool_call", "server_tool_call"):
        return f"{t} {d.tool_name} {d.arguments_json}"
    if t in ("tool_result", "server_tool_result"):
        return f"{t} {d.tool_name} {str(d.result_content)[:60]!r}"
    return t


def blocks(message):
    out = []
    for b in message.content:
        if isinstance(b, TextContent):
            out.append(f"Text({b.text!r})")
        elif isinstance(b, ToolUseContent):
            out.append(f"ToolUse({b.tool_name}, {b.tool_input})")
        elif isinstance(b, ServerToolUseContent):
            out.append(f"ServerToolUse({b.tool_name}, {b.tool_input})")
        elif isinstance(b, ServerToolResultContent):
            out.append(f"ServerToolResult({b.tool_name}, {str(b.tool_result)[:40]!r})")
        else:
            out.append(type(b).__name__)
    return out


def offline_direct_client():
    """The direct client at a dead local address: a fallback that isn't mocked fails fast, offline."""
    return AnthropicProvider(client=anthropic.AsyncAnthropic(api_key="offline", base_url="http://127.0.0.1:9",
                                                             max_retries=0),
                             retry_policy=RetryPolicy(max_retries=1, base_delay=0))


def build_provider(*scripts, store=None, ledger=None, **config):
    """A provider over the fake: each Claude Code process it starts gets the next script."""
    queue = list(scripts)
    config.setdefault("slots", ProcessSlots())
    config.setdefault("meter", PlanMeter())
    return ClaudeCodeProvider(offline_direct_client(), config=ClaudeCodeConfig(workdir=WORKDIR, **config),
                              store=store or MemoryTranscriptStore(), ledger=ledger or MemoryLedgerShown(),
                              client_factory=lambda opts: FakeClaudeCode(opts, queue.pop(0) if queue else ()))


class MemoryLedgerShown(MemoryTranscriptLedger):
    """The in-memory ledger, keeping what it recorded so the notebook can show it."""

    def __init__(self):
        super().__init__()
        self.recorded = []

    async def record(self, **kw):
        self.recorded.append(kw)
        await super().record(**kw)


async def settle(s=0.05):
    await asyncio.sleep(s)


async def step(p, history, *, sink=None, llm_config=None):
    """One model call, as the turn loop makes it."""
    return await p.generate_stream(system_prompt=SYSTEM_PROMPT, messages=history, tool_schemas=SCHEMAS,
                                   llm_config=llm_config or AnthropicLLMConfig(max_tokens=8000), model=MODEL,
                                   sink=sink or Sink(), agent_uuid="notebook-agent")


@tool
def add(a: int, b: int) -> str:
    """Add two numbers together.

    Args:
        a: First number
        b: Second number
    """
    return str(a + b)


@tool
def multiply(a: int, b: int) -> str:
    """Multiply two numbers together.

    Args:
        a: First number
        b: Second number
    """
    return str(a * b)


SAMPLE_TOOLS = [add, multiply]
_registry = ToolRegistry()
_registry.register_tools(SAMPLE_TOOLS)
SCHEMAS = _registry.get_schemas()
SYSTEM_PROMPT = "You are a helpful assistant. Use the tools for arithmetic."
print("tools:", [s.name for s in SCHEMAS])

tools: ['add', 'multiply']


# 1. The seam: a provider value, and an optional lifecycle

`ClaudeCodeProvider` satisfies the library's `Provider` protocol (`agent_base/core/provider.py`), so `AnthropicAgent(provider_value=...)` takes it and nothing else in the loop changes. It wraps an `AnthropicProvider` (the **direct client**) and delegates what is about the Anthropic message shape to it.

It also implements the optional **`ProviderLifecycle`**, which the runtime calls through helpers that treat a missing member as a no-op:

| Member | The runtime calls it | Claude Code does |
|---|---|---|
| `end_run()` | when a run ends: a final reply, an abort or steer, the step limit, an error. Not at a pause | records the run's ledger points, ends the CLI process |
| `aclose()` | when the session ends (eviction, shutdown) | ends the process; waiting tool handlers get "stopped" |
| `side_calls()` | when it builds the compaction controller | a one-shot Claude Code session for the summary call |
| `history_replaced(messages)` | after a compaction that changed the history | ends the process; the next step starts a fresh session |
| `for_child(spec)` | when it builds a sub-agent | a provider of the child's own, on Claude Code too |

In [3]:
from agent_base.core.provider import Provider, ProviderLifecycle

p = build_provider()
print("isinstance(p, Provider)         :", isinstance(p, Provider))
print("isinstance(p, ProviderLifecycle):", isinstance(p, ProviderLifecycle))
print("name                            :", p.name, " <- stamped into agent_config.provider at every run start")
print("inner (direct client)           :", type(p.inner).__name__)
print("delegated                       :", ["default_model", "make_llm_config", "sanitize_chain", "plan_stream_abort",
                                           "extract_tool_calls", "classify_error"])

isinstance(p, Provider)         : True
isinstance(p, ProviderLifecycle): True
name                            : claude_code  <- stamped into agent_config.provider at every run start
inner (direct client)           : AnthropicProvider
delegated                       : ['default_model', 'make_llm_config', 'sanitize_chain', 'plan_stream_abort', 'extract_tool_calls', 'classify_error']


# 2. What Claude Code is given

A run's first model call starts the CLI process with options (`ClaudeAgentOptions`) that make Claude Code a **pure model client**: the system prompt as a file, the library's tools as in-process MCP tools on the server `agent_base`, none of Claude Code's own tools (the web ones only when `server_tools` asks for them), no settings files, `CLAUDE.md` or plugins, no auto-compaction, a permission callback that refuses anything else, an environment with no API key, and the transcript store as the SDK's `session_store`.

In [4]:
FakeClaudeCode.made.clear()
p = build_provider([[text("2 + 3 is 5.")]])
sink = Sink()
task = asyncio.create_task(step(p, [Message.user("What is 2 + 3?")], sink=sink,
                                llm_config=AnthropicLLMConfig(max_tokens=8000, effort="medium")))
await settle()
proc = FakeClaudeCode.made[-1]
o = proc.opts
print("model, effort        :", o.model, o.effort)
print("system_prompt        :", {**o.system_prompt, "path": Path(o.system_prompt["path"]).name})
print("tools                :", o.tools, "<- Claude Code's own tools (none without web server tools)")
print("allowed_tools        :", o.allowed_tools)
print("mcp_servers          :", list(o.mcp_servers), "| strict_mcp_config:", o.strict_mcp_config)
print("setting_sources      :", o.setting_sources, "| plugins:", o.plugins)
print("include_partial_msgs :", o.include_partial_messages, "| max_turns:", o.max_turns)
print("session_store        :", type(o.session_store).__name__)
print("resume / at / fork   :", o.resume, o.resume_session_at, o.fork_session)
print("env API key, output  :", repr(o.env["ANTHROPIC_API_KEY"]), o.env.get("CLAUDE_CODE_MAX_OUTPUT_TOKENS"))
for name in ["mcp__agent_base__add", "Bash", "Write"]:
    verdict = await o.can_use_tool(name, {}, None)
    print(f"  can_use_tool {name:22} -> {type(verdict).__name__}")
print("first message sent   :", proc.sent[0])

model, effort        : claude-opus-5-5 medium
system_prompt        : {'type': 'file', 'path': 'notebook-agent.txt'}
tools                : [] <- Claude Code's own tools (none without web server tools)
allowed_tools        : ['mcp__agent_base__add', 'mcp__agent_base__multiply']
mcp_servers          : ['agent_base'] | strict_mcp_config: True
setting_sources      : [] | plugins: []
include_partial_msgs : True | max_turns: 1000
session_store        : MemoryTranscriptStore
resume / at / fork   : None None False
env API key, output  : '' 8000
  can_use_tool mcp__agent_base__add   -> PermissionResultAllow
  can_use_tool Bash                   -> PermissionResultDeny
  can_use_tool Write                  -> PermissionResultDeny
first message sent   : [{'type': 'text', 'text': 'What is 2 + 3?'}]


# 3. One model call answers one `generate_stream`

Claude Code's raw Anthropic events go through **the library's own stream translator**, so the frames are the ones the direct client emits. The same events build the response message: the model's own ids, thinking blocks, and usage with the cache-write split in `raw_usage`. The process stays open after a final reply (an end-of-turn hook may retry); `end_run` closes it, after the run's ledger points are recorded.

In [5]:
turn = await asyncio.wait_for(task, 5)
print("frames      :", [render(d) for d in sink.deltas])
print("message     :", blocks(turn.message), "| provider:", turn.message.provider)
print("usage       :", turn.message.usage.output_tokens, "out,", turn.message.usage.cache_write_tokens, "cache write")
print("raw_usage   :", turn.message.usage.raw_usage["cache_creation"])
print("still open after the final reply:", not proc.closed)
await p.end_run()
print("after end_run: closed", proc.closed, "| ledger points", len(p.ledger.recorded))

frames      : ["text '2 + 3 is 5.'", 'text (block closed)']
message     : ["Text('2 + 3 is 5.')"] | provider: claude_code
usage       : 10 out, 100 cache write
raw_usage   : {'ephemeral_1h_input_tokens': 100, 'ephemeral_5m_input_tokens': 0}
still open after the final reply: True
after end_run: closed True | ledger points 1


# 4. A tool call: the handler that waits

1. The model asks for `mcp__agent_base__add`. The provider strips the prefix, so the loop sees `add`, with the **model's own tool-call id**.
2. Claude Code calls the MCP handler, which finds its call by the id in the request's `_meta` (`claudecode/toolUseId`) and **waits**.
3. The library's tool round runs `add` and appends the result.
4. The next `generate_stream` resolves the waiting handler; Claude Code sends the result to the model as a proper `tool_result` and streams the next model call on the **same process**.

In [6]:
FakeClaudeCode.made.clear()
p = build_provider([[text("Adding."), call("add", a=2, b=3)], [text("2 + 3 = 5.")]])
history = [Message.user("What is 2 + 3?")]
turn1 = await step(p, history)
proc = FakeClaudeCode.made[-1]
await settle()
print("step 1     :", blocks(turn1.message))
print("waiting    :", list(p._futures), "| Claude Code's call done?", proc.tool_calls[0].done())
use = next(b for b in turn1.message.content if isinstance(b, ToolUseContent))
history += [turn1.message, Message.user([ToolResultContent(tool_name="add", tool_id=use.tool_id, tool_result=add(a=2, b=3))])]
turn2 = await step(p, history)
print("handler got:", (await proc.tool_calls[0])[0].root.content[0].text)
print("step 2     :", blocks(turn2.message))
print("messages sent to Claude Code:", len(proc.sent), "| processes:", p.processes_started)
await p.end_run()

step 1     : ["Text('Adding.')", "ToolUse(add, {'a': 2, 'b': 3})"]
waiting    : ['toolu_2'] | Claude Code's call done? False
handler got: 5
step 2     : ["Text('2 + 3 = 5.')"]
messages sent to Claude Code: 1 | processes: 1


# 5. The whole loop, offline: `AnthropicAgent(provider_value=...)`

Nothing is wired by hand: the runtime calls the provider's lifecycle. Run 1 starts a session and closes its process at the run's end. Run 2 **resumes** the transcript at the ledger point that matches the history. Run 3 is a cold load (a new agent object and a new provider over the same storage): it resumes the same way.

In [7]:
from agent_base.storage import create_adapters

FakeClaudeCode.made.clear()
STORE, LEDGER = MemoryTranscriptStore(), MemoryLedgerShown()
ADAPTERS = dict(zip(("config_adapter", "conversation_adapter", "run_adapter"),
                    create_adapters("filesystem", base_path=tempfile.mkdtemp(prefix="cc_nb_"))))


def make_agent(provider, **kw):
    return AnthropicAgent(system_prompt=SYSTEM_PROMPT, model=MODEL, config=AnthropicLLMConfig(max_tokens=8000),
                          tools=SAMPLE_TOOLS, provider_value=provider, **ADAPTERS, **kw)


p1 = build_provider([[call("add", a=2, b=3)], [call("multiply", a=5, b=10)], [text("(2 + 3) × 10 = 50.")]],
                    [[text("You asked for (2 + 3) × 10, which is 50.")]], store=STORE, ledger=LEDGER)
agent = make_agent(p1)
r1 = await agent.run("What is (2 + 3) * 10?")
print("run 1:", r1.final_answer, "| steps", r1.total_steps, "| provider", agent.agent_config.provider,
      "| processes", p1.processes_started, "| closed", FakeClaudeCode.made[-1].closed)
print("ledger:", [(r["prefix_len"], r["session_id"]) for r in LEDGER.recorded])
r2 = await agent.run("What did I ask you before?")
proc = FakeClaudeCode.made[-1]
print("run 2:", r2.final_answer, "| resumed", proc.opts.resume, "fork", proc.opts.fork_session, "| sent", proc.sent)

cold = make_agent(build_provider([[text("Still 50.")]], store=STORE, ledger=LEDGER), agent_uuid=agent.agent_uuid)
r3 = await cold.run("Remind me of the answer.")
print("run 3 (cold load):", r3.final_answer, "| resumed", FakeClaudeCode.made[-1].opts.resume,
      "| fallen back", cold.provider.fallen_back)

run 1: (2 + 3) × 10 = 50. | steps 3 | provider claude_code | processes 1 | closed True
ledger: [(2, 'cc-b26f5a69'), (4, 'cc-b26f5a69'), (6, 'cc-b26f5a69')]
run 2: You asked for (2 + 3) × 10, which is 50. | resumed cc-b26f5a69 fork False | sent [[{'type': 'text', 'text': 'What did I ask you before?'}]]


run 3 (cold load): Still 50. | resumed cc-b26f5a69 | fallen back False


# 6. Fingerprints: finding a point of the history again

The library's history is the one that counts; Claude Code's transcript is a copy the CLI needs to resume. The ledger maps the fingerprint of each point where a model call ended to the transcript entry holding the same point. A message's hash covers its role and content in the formatter's wire form; a prefix's fingerprint chains the hashes, so it names one exact conversation so far.

In [8]:
fmt = p1.formatter
h = agent.agent_config.context_messages
fps = prefix_fingerprints(fmt, h)
for i, (m, fp) in enumerate(zip(h, fps)):
    print(f"  [{i}] {m.role.value:9} {fp[:16]}…  {blocks(m) if m.role.value == 'assistant' else ''}")
reloaded = [Message.from_dict(json.loads(json.dumps(m.to_dict()))) for m in h]
print("same after a JSON round trip      :", prefix_fingerprints(fmt, reloaded) == fps)
edited = [Message.user("What is (2 + 4) * 10?"), *h[1:]]
print("an edit to message 0 changes all  :", all(a != b for a, b in zip(prefix_fingerprints(fmt, edited), fps)))

  [0] user      5a44e7a09385704f…  
  [1] assistant 43bf0f21c29de2a4…  ["ToolUse(add, {'a': 2, 'b': 3})"]
  [2] user      7b1b98a07962b8ae…  
  [3] assistant a3ab8738d590a174…  ["ToolUse(multiply, {'a': 5, 'b': 10})"]
  [4] user      57726fa29cc6cf22…  
  [5] assistant 3f60a4c7b338c8df…  ["Text('(2 + 3) × 10 = 50.')"]
  [6] user      c3846d2a753708a4…  
  [7] assistant 6aadf753fcd5bc94…  ["Text('You asked for (2 + 3) × 10, which is 50.')"]
same after a JSON round trip      : True
an edit to message 0 changes all  : True


# 7. Where a run's process resumes

| The history before this run | What the provider does |
|---|---|
| Empty | A new Claude Code session |
| A point at a transcript's end | Resume that transcript |
| A point before a transcript's end (a reset, a fork) | Resume at that entry and branch (`resume_session_at` + `fork_session`) |
| Beyond the last point (an abort mid-step, a steer, a parked run across a restart) | Resume at the last point; what follows goes in **as text**, in order |
| No point at all (a reply cut before its point was recorded, an edited history, a session that ran on the direct client), or a ledger that fails | A fresh session takes the whole history **as text**, as after a compaction: the session stays on Claude Code |

In [9]:
FakeClaudeCode.made.clear()
point = [Message.user("List the files."),
         Message.assistant([ToolUseContent(tool_name="add", tool_id="toolu_a", tool_input={"a": 1, "b": 1})])]
cut = Message.user([ToolResultContent(tool_name="add", tool_id="toolu_a", tool_result="aborted")])


async def first_step(history, transcript_end):
    store, ledger = MemoryTranscriptStore(), MemoryLedgerShown()
    await ledger.record(agent_uuid="a", fingerprint=prefix_fingerprints(fmt, point)[-1], prefix_len=2,
                        session_id="cc-0", entry_uuid="u-2")
    await store.append({"project_key": project_key_for_directory(str(WORKDIR.resolve())), "session_id": "cc-0"},
                       [{"type": "assistant", "uuid": transcript_end}])
    p = build_provider([[text("ok")]], store=store, ledger=ledger)
    await step(p, history)
    proc = FakeClaudeCode.made[-1]
    await p.end_run()
    return proc.opts, [b.get("text", f"<{b['type']}>")[:70] for b in proc.sent[0]]


cases = {
    "plain resume": (point + [Message.user("And again?")], "u-2"),
    "reset / fork (branch)": (point + [Message.user("And again?")], "u-7"),
    "beyond the point": (point + [cut, Message.user("Never mind. 17 × 3?")], "u-2"),
    "reply cut off": (point + [cut, Message.assistant([TextContent(text="Building the table")]),
                               Message.user("Use only a.csv.")], "u-2"),
}
cases["no point"] = ([Message.user("Hi."), Message.assistant("Hello."), Message.user("And again?")], "u-2")
for name, (history, end) in cases.items():
    opts, sent = await first_step(history, end)
    print(f"{name}\n  resume={opts.resume} at={opts.resume_session_at} fork={opts.fork_session}")
    for s in sent:
        print("   sent:", s)

plain resume
  resume=cc-0 at=None fork=False
   sent: And again?


reset / fork (branch)
  resume=cc-0 at=u-2 fork=True
   sent: And again?


beyond the point
  resume=cc-0 at=None fork=False
   sent: Results of the tool calls from your last step. The step was cut off be
   sent: <tool_result tool_use_id="toolu_a" name="add" is_error="false">
   sent: aborted
   sent: </tool_result>
   sent: Never mind. 17 × 3?


reply cut off
  resume=cc-0 at=None fork=False
   sent: Your previous turn was cut off before it finished (the user stopped or
   sent: <tool_result tool_use_id="toolu_a" name="add" is_error="false">
   sent: aborted
   sent: </tool_result>
   sent: <your_reply>Building the table</your_reply>
   sent: Use only a.csv.


2026-11-11T18:32:02.997940Z [info     ] claude_code_fresh_session      [agent_base.providers.claude_code.provider] agent_uuid=notebook-agent reason='no transcript point matches the history'


no point
  resume=None at=None fork=False
   sent: This conversation continues in a new session. Everything before this m
   sent: <user>
   sent: Hi.
   sent: </user>
   sent: <your_reply>Hello.</your_reply>
   sent: And again?


# 8. The fallback: the direct client, for good

A session moves to the direct client when Claude Code can't carry it on: a CLI or login failure (the process ending mid-step, the CLI ending the query itself), the plan's refusal, or no free process slot. Before any frame reached a client, the same step runs direct; after frames went out, the step fails and the next one goes direct. `name` becomes `anthropic`. With `fallback="off"` nothing ever reaches an API key. A history with no ledger point is not a fallback (section 7). The direct client is mocked here.

In [10]:
from unittest.mock import AsyncMock
from claude_agent_sdk import RateLimitEvent, RateLimitInfo

REFUSED = RateLimitEvent(rate_limit_info=RateLimitInfo(status="rejected", rate_limit_type="five_hour"),
                         uuid="r1", session_id="cc-x")


async def refused_step(p):
    task = asyncio.create_task(step(p, [Message.user("Hi.")]))
    await settle()
    FakeClaudeCode.made[-1].push(REFUSED)
    return await asyncio.wait_for(task, 5)

FakeClaudeCode.made.clear()
p = build_provider()
p.inner.generate_stream = AsyncMock(return_value="<the direct client's turn>")
print("plan limit refused :", await refused_step(p), "| name:", p.name, "| reason:", p.fallback_reason)
print("next step          :", await step(p, [Message.user("Hi.")]), "(the session stays on the direct client)")

p = build_provider(fallback="off")
p.inner.generate_stream = AsyncMock()
try:
    await refused_step(p)
except RuntimeError as e:
    print("fallback off       :", e, "| direct client called:", p.inner.generate_stream.called)
await p.aclose()

2026-11-11T18:32:03.767238Z [warning  ] claude_code_plan_near_limit    [agent_base.providers.claude_code.provider] status=rejected until=1794422523 utilization=None window=five_hour


2026-11-11T18:32:03.768854Z [warning  ] claude_code_fallback           [agent_base.providers.claude_code.provider] agent_uuid=notebook-agent reason="the plan's five_hour limit refused the call"


plan limit refused : <the direct client's turn> | name: anthropic | reason: the plan's five_hour limit refused the call
next step          : <the direct client's turn> (the session stays on the direct client)


2026-11-11T18:32:04.420048Z [warning  ] claude_code_plan_near_limit    [agent_base.providers.claude_code.provider] status=rejected until=1794422524 utilization=None window=five_hour


2026-11-11T18:32:04.420048Z [error    ] claude_code_step_failed_no_fallback [agent_base.providers.claude_code.provider] reason="the plan's five_hour limit refused the call"


fallback off       : Claude Code could not carry this step: the plan's five_hour limit refused the call | direct client called: False


# 9. An error from the model API is not a fallback

When the API fails a call, the CLI says so in the query's result (`is_error`, `terminal_reason`, `api_error_status`). The step fails with a typed `ProviderError`, as on the direct client, and the session stays on Claude Code with either `fallback` setting. The process ends; the run keeps its slot for the next attempt.

| The CLI reports | The step fails with |
|---|---|
| `prompt_too_long`, its own `blocking_limit`, HTTP 413 | `CONTEXT_OVERFLOW`: the turn loop compacts and retries, when compaction is on (section 12) |
| HTTP 5xx, `server_error` (after the CLI's own retries) | `PROVIDER_OVERLOADED`, retriable |
| any other refusal (`invalid_request`, `unknown`) | `PROVIDER_STATUS` |

The CLI ending the query itself (`error_max_turns`), or its process ending mid-step, is a failure of Claude Code: the fallback. No failure leaves a step waiting.

**A reply cut off mid-stream** (a server error, a lost connection) isn't retried from scratch by the CLI: it keeps the reply's finished blocks, drops the block cut midway, and sends its own error message. Then it runs the reply's tool calls, or, with text only, asks the model to resume in a new model call of the same query. The step follows that shape: a finished tool call ends the step there; with text only, the resumed call folds into the same step (second cell).

In [11]:
from agent_base.core.errors import ErrorCode
from agent_base.core.provider import ProviderError

for label, item in [
    ("prompt too long ", failure("Prompt is too long", reason="prompt_too_long", status=400)),
    ("API overloaded  ", failure("API Error: Overloaded", error="server_error", status=529)),
    ("bad request     ", failure("API Error: messages.0: bad image", status=400)),
]:
    p = build_provider([item])
    p.inner.generate_stream = AsyncMock()
    try:
        await asyncio.wait_for(step(p, [Message.user("Hi.")]), 5)
    except ProviderError as e:
        print(f"{label}: {e.code.value:19} retriable={e.retriable!s:5} | still on Claude Code: {not p.fallen_back}"
              f" | slot kept: {p.config.slots.holds(p)} | direct client called: {p.inner.generate_stream.called}")

p = build_provider([failure("Reached maximum number of turns (1000)", error=None, subtype="error_max_turns")])
p.inner.generate_stream = AsyncMock(return_value="<the direct client's turn>")
print("CLI ended query :", await asyncio.wait_for(step(p, [Message.user("Hi.")]), 5), "| reason:", p.fallback_reason)

FakeClaudeCode.made.clear()
p = build_provider()  # this process never answers
p.inner.generate_stream = AsyncMock(return_value="<the direct client's turn>")
task = asyncio.create_task(step(p, [Message.user("Hi.")]))
await settle()
FakeClaudeCode.made[-1].push(None)  # the CLI exits mid-step: its message stream ends
print("process ended   :", await asyncio.wait_for(task, 5), "| reason:", p.fallback_reason)

2026-11-11T18:32:05.017103Z [warning  ] claude_code_step_failed        [agent_base.providers.claude_code.provider] code=context_overflow message='Prompt is too long' native_code=prompt_too_long


prompt too long : context_overflow    retriable=False | still on Claude Code: True | slot kept: True | direct client called: False


2026-11-11T18:32:05.593396Z [warning  ] claude_code_step_failed        [agent_base.providers.claude_code.provider] code=provider_overloaded message='API Error: Overloaded' native_code=529


API overloaded  : provider_overloaded retriable=True  | still on Claude Code: True | slot kept: True | direct client called: False


2026-11-11T18:32:06.182460Z [warning  ] claude_code_step_failed        [agent_base.providers.claude_code.provider] code=provider_status message='API Error: messages.0: bad image' native_code=400


bad request     : provider_status     retriable=False | still on Claude Code: True | slot kept: True | direct client called: False


2026-11-11T18:32:06.801591Z [warning  ] claude_code_fallback           [agent_base.providers.claude_code.provider] agent_uuid=notebook-agent reason='Claude Code ended the query (error_max_turns): Reached maximum number of turns (1000)'


CLI ended query : <the direct client's turn> | reason: Claude Code ended the query (error_max_turns): Reached maximum number of turns (1000)


2026-11-11T18:32:07.422113Z [warning  ] claude_code_fallback           [agent_base.providers.claude_code.provider] agent_uuid=notebook-agent reason='the Claude Code process ended'


process ended   : <the direct client's turn> | reason: the Claude Code process ended


In [12]:
FakeClaudeCode.made.clear()
p = build_provider([cut_reply([text("Here is the plan.")], "Step one: re"), [text("Step one: read the file.")]])
sink = Sink()
turn = await asyncio.wait_for(step(p, [Message.user("Plan it.")], sink=sink), 5)
print("text resumed    : streamed", [render(d) for d in sink.deltas if getattr(d, "type", "") == "text"])
print("                  message ", blocks(turn.message), "| stop", turn.message.stop_reason)
await p.aclose()

FakeClaudeCode.made.clear()
p = build_provider([cut_reply([text("Adding."), call("add", a=2, b=3)], "Then I will"), [text("2 + 3 = 5.")]])
turn = await asyncio.wait_for(step(p, [Message.user("What is 2 + 3?")]), 5)
print("tool call kept  : message ", blocks(turn.message), "| stop", turn.message.stop_reason,
      "| on Claude Code:", not p.fallen_back)
await p.aclose()

2026-11-11T18:32:08.139986Z [warning  ] claude_code_reply_cut          [agent_base.providers.claude_code.provider] finished_blocks=1 kind=server_error


text resumed    : streamed ["text 'Here is the plan.'", 'text (block closed)', "text 'Step one: re'", "text 'Step one: read the file.'", 'text (block closed)']
                  message  ["Text('Here is the plan.')", "Text('Step one: read the file.')"] | stop end_turn


2026-11-11T18:32:08.709775Z [warning  ] claude_code_reply_cut          [agent_base.providers.claude_code.provider] finished_blocks=2 kind=server_error


tool call kept  : message  ["Text('Adding.')", "ToolUse(add, {'a': 2, 'b': 3})"] | stop tool_use | on Claude Code: True


# 10. Web search and fetch

Anthropic's server tools can't pass through the CLI, so Claude Code's own `WebSearch` and `WebFetch` stand in, on when the request's `server_tools` name `web_search` / `web_fetch`. They are framed as server tools and folded into one step; the direct client gets them as text. A `PreToolUse` hook holds each to its `max_uses` per step, and every `WebFetch` URL passes the URL guard first, because `WebFetch` runs on the host.

In [13]:
FakeClaudeCode.made.clear()
cfg = AnthropicLLMConfig(max_tokens=8000, server_tools=[
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 2},
    {"type": "web_fetch_20250910", "name": "web_fetch"}])
p = build_provider([[web("WebSearch", "Repo rate 5.50% (6 Nov 2026)", query="RBI repo rate")],
                    [text("The repo rate is 5.50%.")]])
sink = Sink()
turn = await step(p, [Message.user("What is the RBI repo rate?")], sink=sink, llm_config=cfg)
print("tools on   :", FakeClaudeCode.made[-1].opts.tools)
print("message    :", blocks(turn.message))
print("frames     :", [render(d) for d in sink.deltas if getattr(d, "type", "").startswith("server_")])
print("usage      : output", turn.message.usage.output_tokens, "(two model calls, summed)")
print("for direct :", [b.text[:40] for b in _plain_history([turn.message])[0].content])

p._web_uses = {}
for name, inp in [("WebSearch", {"query": "a"}), ("WebSearch", {"query": "b"}), ("WebSearch", {"query": "c"}),
                  ("WebFetch", {"url": "http://169.254.169.254/latest/meta-data"}), ("WebFetch", {"url": "http://localhost:8000"})]:
    out = await p._web_hook({"tool_name": name, "tool_input": inp}, "id", None)
    d = out.get("hookSpecificOutput", {})
    print(f"  {name:9} {json.dumps(inp):52} -> {d.get('permissionDecision', 'allow')}  {d.get('permissionDecisionReason', '')}")
await p.end_run()

2026-11-11T18:32:09.386484Z [info     ] claude_code_web_tool_limit     [agent_base.providers.claude_code.provider] limit=2 tool=WebSearch


2026-11-11T18:32:09.402904Z [warning  ] claude_code_web_fetch_refused  [agent_base.providers.claude_code.provider] reason='169.254.169.254 is not on the public internet.'


2026-11-11T18:32:09.411507Z [warning  ] claude_code_web_fetch_refused  [agent_base.providers.claude_code.provider] reason='localhost is not on the public internet.'


tools on   : ['WebSearch', 'WebFetch']
message    : ["ServerToolUse(WebSearch, {'query': 'RBI repo rate'})", "ServerToolResult(WebSearch, 'Repo rate 5.50% (6 Nov 2026)')", "Text('The repo rate is 5.50%.')"]
frames     : ['server_tool_call WebSearch {"query": "RBI repo rate"}', "server_tool_result web_tool_result 'Repo rate 5.50% (6 Nov 2026)'"]
usage      : output 20 (two model calls, summed)
for direct : ['[WebSearch {"query": "RBI repo rate"}]', '[WebSearch result]\nRepo rate 5.50% (6 No', 'The repo rate is 5.50%.']
  WebSearch {"query": "a"}                                       -> allow  
  WebSearch {"query": "b"}                                       -> allow  
  WebSearch {"query": "c"}                                       -> deny  WebSearch can be used at most 2 times in one step. Work with the results you already have.
  WebFetch  {"url": "http://169.254.169.254/latest/meta-data"}   -> deny  169.254.169.254 is not on the public internet.
  WebFetch  {"url": "http://localhost:

# 11. The CLI's environment, the process slots, the plan's meter

- **The environment** carries no secret but the login: every variable whose name matches `env_blank_patterns` is blanked, `ANTHROPIC_API_KEY` is empty, `CLAUDE_CODE_OAUTH_TOKEN` is kept.
- **Process slots:** one process per run in flight, parked runs included; at most `max_processes` at once.
- **The plan's meter:** Claude Code reports rate-limit events; a host can ask `near_limit()` before giving a new session Claude Code.

In [14]:
import time

demo = {"DEMO_API_KEY": "x", "DEMO_DATABASE_URL": "x", "AWS_DEMO": "x", "CLAUDE_CODE_OAUTH_TOKEN": "x", "DEMO_PLAIN": "x"}
saved = {k: os.environ.get(k) for k in demo}
os.environ.update(demo)
try:
    env = cli_env()
    for k in demo:
        print(f"  {k:24} -> {'blanked' if env.get(k) == '' else 'inherited unchanged'}")
finally:
    for k, v in saved.items():
        os.environ.pop(k) if v is None else os.environ.__setitem__(k, v)

s, a, b = ProcessSlots(), object(), object()
print("\nA acquires (limit 1)  :", await s.acquire(a, limit=1, wait_s=1))
print("B waits 0.2 s         :", await s.acquire(b, limit=1, wait_s=0.2))
meter, now = PlanMeter(), time.time()
meter.note(SimpleNamespace(status="allowed_warning", rate_limit_type="five_hour", utilization=0.86, resets_at=now + 3600), now=now)
print("near_limit()          :", meter.near_limit(now=now))

  DEMO_API_KEY             -> blanked
  DEMO_DATABASE_URL        -> blanked
  AWS_DEMO                 -> blanked
  CLAUDE_CODE_OAUTH_TOKEN  -> inherited unchanged
  DEMO_PLAIN               -> inherited unchanged

A acquires (limit 1)  : True


B waits 0.2 s         : False
near_limit()          : LimitState(window='five_hour', status='allowed_warning', utilization=0.86, until=1794425529.6416411)


# 12. Compaction stays on Claude Code

Compaction is the library's own, on only with a `CompactionConfig`. On Claude Code:

1. The summary call goes to `provider.side_calls()`: a **one-shot** Claude Code session with no ledger and no tools; the run waits for it.
2. When compaction changed the history, the runtime calls `history_replaced()`: the run's process, which holds the long transcript, closes.
3. The next step starts a **fresh session** with the compacted history as text, in order, with the step's new message. Its ledger point covers the compacted history, so later runs and a cold load resume it.

A `before_compact` hook allows a single compaction here, so the run after it shows the resume.

Around those steps:

- **Both triggers work:** past `threshold_tokens` before a step, and a context too long that Claude Code reports (section 9), after which the step is retried on the fresh session.
- **The summary runs on the run's slot.** If the run's process is open, it ends first, once its ledger points are recorded, so a session never holds two processes.
- **A failed summary falls back for that call only.** The next summary tries Claude Code again; while the session is on the direct client, summaries go direct too.

The second cell runs the overflow path with `max_processes=1` and `fallback="off"`: a second slot, or any fallback, would fail it.

In [15]:
from agent_base.core.hooks import HookMatcher, HookOutcome
from agent_base.providers.anthropic.compaction import CompactionConfig

FakeClaudeCode.made.clear()
STORE11, LEDGER11 = MemoryTranscriptStore(), MemoryLedgerShown()
ADAPTERS11 = dict(zip(("config_adapter", "conversation_adapter", "run_adapter"),
                      create_adapters("filesystem", base_path=tempfile.mkdtemp(prefix="cc_nb11_"))))
compacted = []


async def once(ctx):
    return HookOutcome(decision="block", reason="one compaction is enough here") if compacted else None


async def note(ctx):
    stats = ctx.stats if isinstance(ctx.stats, dict) else vars(ctx.stats)
    if stats["messages_after"] < stats["messages_before"]:  # after_compact also fires when nothing changed
        compacted.append(stats)


def compacting_agent(provider, **kw):
    return AnthropicAgent(system_prompt=SYSTEM_PROMPT, model=MODEL, config=AnthropicLLMConfig(max_tokens=8000),
                          tools=SAMPLE_TOOLS, provider_value=provider, **ADAPTERS11,
                          compaction_config=CompactionConfig(threshold_tokens=1, preserve_recent_tokens=1),
                          hooks={"before_compact": [HookMatcher(matcher=None, hooks=[once])],
                                 "after_compact": [HookMatcher(matcher=None, hooks=[note])]}, **kw)


p11 = build_provider(
    [[call("add", a=2, b=3)], [call("multiply", a=5, b=10)], [text("(2 + 3) × 10 = 50.")]],  # run 1
    [[text("Summary: the user asked for (2 + 3) × 10; it is 50.")]],                         # the side call
    [[text("Plus 7 is 57.")]],                                                                # run 2, fresh session
    store=STORE11, ledger=LEDGER11)
agent11 = compacting_agent(p11)
await agent11.run("What is (2 + 3) * 10?")
r = await agent11.run("Now add 7.")
run1, side, run2 = FakeClaudeCode.made
print("side call   : resume", side.opts.resume, "| store", side.opts.session_store, "| tools", side.opts.allowed_tools)
print("              last block sent:", side.sent[0][-1]["text"][:60], "…")
print("run 2       : fresh session (resume", run2.opts.resume, ")")
for b in run2.sent[0]:
    print("   sent:", b.get("text", "")[:80])
print("run 2 final :", r.final_answer, "| fallen back:", p11.fallen_back)
print("context now :", [(m.role.value, (blocks(m) or ["<tool results>"])[0][:40]) for m in agent11.agent_config.context_messages])

cold = compacting_agent(build_provider([[text("It was 57.")]], store=STORE11, ledger=LEDGER11), agent_uuid=agent11.agent_uuid)
r3 = await cold.run("What was the last answer?")
print("cold load   : resumed", FakeClaudeCode.made[-1].opts.resume, "(the re-seeded session:",
      FakeClaudeCode.made[-1].opts.resume == run2.session_id, ") | fallen back:", cold.provider.fallen_back)

side call   : resume None | store None | tools []
              last block sent: Summarize the earlier conversation history above.

Preserve: …
run 2       : fresh session (resume None )
   sent: This conversation continues in a new session. Everything before this message, in
   sent: <your_reply>Summary: the user asked for (2 + 3) × 10; it is 50.</your_reply>
   sent: Now add 7.
run 2 final : Plus 7 is 57. | fallen back: False
context now : [('assistant', "Text('Summary: the user asked for (2 + 3"), ('user', "Text('Now add 7.')"), ('assistant', "Text('Plus 7 is 57.')")]


cold load   : resumed cc-0c15f22c (the re-seeded session: True ) | fallen back: False


In [16]:
FakeClaudeCode.made.clear()
p12 = build_provider(
    [[text("Hello.")]],                                                     # run 1
    [failure("Prompt is too long", reason="prompt_too_long", status=400)],  # run 2: too long to resume
    [[text("Summary: the user said hello.")]],                              # the summary call
    [[text("Hello again.")]],                                               # run 2, retried on a fresh session
    max_processes=1, slot_wait_s=0.2, fallback="off")
agent12 = AnthropicAgent(system_prompt=SYSTEM_PROMPT, model=MODEL, config=AnthropicLLMConfig(max_tokens=8000),
                         tools=SAMPLE_TOOLS, provider_value=p12,
                         compaction_config=CompactionConfig(threshold_tokens=10**9, preserve_recent_tokens=1))
await agent12.run("Hi.")
r = await agent12.run("Hi again.")
for name, f in zip(("run 1", "run 2, too long", "summary", "run 2, retried"), FakeClaudeCode.made):
    print(f"{name:15} : resume {f.opts.resume!s:11} | processes open as it started: {f.open_at_start} | closed: {f.closed}")
print("retried, sent   :", [b.get("text", "")[:50] for b in FakeClaudeCode.made[-1].sent[0]])
print("answer          :", r.final_answer, "| fallen back:", p12.fallen_back,
      "| slots in use now:", p12.config.slots.in_use)

# A summary that can't get a slot falls back for that call only.
p = build_provider([[text("Summary.")]], max_processes=1, slot_wait_s=0.05)
p.inner.generate = AsyncMock(return_value="<the direct client's summary>")
side, other = p.side_calls(), object()
chain = [Message.user("Hi."), Message.assistant("Hello."), Message.user("Summarize.")]


async def summarise():
    return await side.generate(system_prompt=None, messages=chain, tool_schemas=[], llm_config=AnthropicLLMConfig(),
                               model=MODEL, agent_uuid="notebook-agent")

await p.config.slots.acquire(other, limit=1, wait_s=0)  # another session holds the only slot
print("summary, no slot:", await summarise(), "| session fallen back:", p.fallen_back)
p.config.slots.release(other)
print("next summary    :", blocks((await summarise()).message), "| on Claude Code, session fallen back:", p.fallen_back)

2026-11-11T18:32:11.702339Z [warning  ] claude_code_step_failed        [agent_base.providers.claude_code.provider] code=context_overflow message='Prompt is too long' native_code=prompt_too_long


run 1           : resume None        | processes open as it started: 1 | closed: True
run 2, too long : resume cc-26200d43 | processes open as it started: 1 | closed: True
summary         : resume None        | processes open as it started: 1 | closed: True
run 2, retried  : resume None        | processes open as it started: 1 | closed: True
retried, sent   : ['This conversation continues in a new session. Ever', '<your_reply>Summary: the user said hello.</your_re', 'Hi again.']
answer          : Hello again. | fallen back: False | slots in use now: 0


2026-11-11T18:32:12.396088Z [warning  ] claude_code_processes_full     [agent_base.providers.claude_code.provider] limit=1 wait_s=0.05


2026-11-11T18:32:12.397101Z [warning  ] claude_code_side_call_fallback [agent_base.providers.claude_code.provider] reason='all 1 Claude Code processes were in use for 0.05 s'


summary, no slot: <the direct client's summary> | session fallen back: False
next summary    : ["Text('Summary.')"] | on Claude Code, session fallen back: False


# 13. Sub-agents follow the parent onto Claude Code

`SubAgentTool` asks the parent's provider for the child's (`for_child`). Claude Code returns a provider of the child's own (its own process, counted in the same slots) that shares the parent's store, ledger and meter, so a child's model calls stay on Claude Code instead of reaching the API key. A provider without `for_child` keeps today's default, an `AnthropicProvider`.

In [17]:
from agent_base.common_tools.sub_agent_tool import SubAgentParentContext, SubAgentSpec, SubAgentTool

parent = make_agent(build_provider(store=STORE, ledger=LEDGER))
spec = SubAgentSpec(name="helper", description="Helps with arithmetic.", system_prompt="Help.", model=MODEL)
child = SubAgentTool({"helper": spec})._default_child_agent_builder(spec, None, SubAgentParentContext(parent_agent=parent))
print("child's provider        :", type(child.provider).__name__, "| its own:", child.provider is not parent.provider)
print("shares store and ledger :", child.provider.store is parent.provider.store and child.provider.ledger is parent.provider.ledger)

child's provider        : ClaudeCodeProvider | its own: True
shares store and ledger : True


# 14. Live: a real run on your Claude Code sign-in (opt-in)

Everything above ran against the fake. This runs the real CLI bundled with the SDK on **your own Claude Code sign-in** (the CLI's environment has no API key). It draws on your plan's limits, so it is off by default: set `RUN_LIVE = True`. Two runs on one agent: the first starts a session and calls the tools; the second resumes it from the ledger.

A claude.ai plan's login may not be used in a third-party product without Anthropic's prior approval (Claude Agent SDK overview).

In [18]:
RUN_LIVE = False
LIVE_MODEL = "claude-haiku-5-5"

if not RUN_LIVE:
    print("Skipped: set RUN_LIVE = True to run on your Claude Code sign-in.")
else:
    live = ClaudeCodeProvider(config=ClaudeCodeConfig(workdir=WORKDIR, fallback="off"))
    live_agent = AnthropicAgent(system_prompt=SYSTEM_PROMPT, model=LIVE_MODEL, tools=SAMPLE_TOOLS,
                                config=AnthropicLLMConfig(max_tokens=4000), provider_value=live)
    lr1 = await live_agent.run("Use the tools to work out (2 + 3) * 10. Answer in one sentence.")
    print("run 1:", lr1.final_answer, "| steps:", lr1.total_steps, "| provider:", live_agent.agent_config.provider)
    lr2 = await live_agent.run("What was the first number you added?")
    print("run 2:", lr2.final_answer, "| processes started:", live.processes_started, "| fallen back:", live.fallen_back)
    await live_agent.aclose()

Skipped: set RUN_LIVE = True to run on your Claude Code sign-in.
